In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import scipy.stats as stats
#import openassetpricing as oap
import gc

In [2]:
#载入数据
returns_df = pd.read_csv("Stock_return.csv")
print(returns_df.head(10))

#日期格式化，将monthend转换为标准的datetime格式
returns_df['monthend'] = pd.to_datetime(returns_df['monthend'], format='%Y%m%d')
returns_df['monthend'] = returns_df['monthend'] + pd.offsets.MonthEnd(0)

#处理 CRSP 的特殊价格：负数价格表示当天没有交易，取的是买卖报价的平均值 (好像没有负数价格，但还是处理一下)
#我们需要取绝对值来计算市值
returns_df['prc'] = returns_df['prc'].abs()

#补齐特征1.计算月末市值 (Market Equity)
#SHROUT 通常是以千股为单位，为了严谨可以保持原样或乘 1000，这里保持原样计算相对市值即可
returns_df['mvel'] = returns_df['prc'] * returns_df['SHROUT']

#补齐特征2. 价格
# 论文中通常使用对数价格或原始价格的某种变换，这里先保留原始 prc
returns_df['price'] = returns_df['prc']

# 补齐特征 3：短期反转 (Short-term Reversal / mom1m)
# 短期反转就是过去 1 个月的收益率，我们需要按照 PERMNO 排序后向下平移一行 (shift)
returns_df = returns_df.sort_values(['PERMNO', 'monthend'])
returns_df['mom1m'] = returns_df.groupby('PERMNO')['exret'].shift(1)

# 删除无法计算短期反转的起始行
returns_df = returns_df.dropna(subset=['mom1m', 'exret'])

#剔除目标变量 exret 为缺失值的行（没有目标变量就无法训练）
returns_df = returns_df.dropna(subset=['exret'])

#保留我们后续合并需要的核心列
returns_df = returns_df[['PERMNO', 'monthend', 'exret', 'mvel', 'VOL', 'price', 'mom1m']]

print(f"清洗后的收益率数据维度: {returns_df.shape}")

returns_df

   PERMNO  VOL  SHROUT  monthend     exret       prc
0   10001  168    2281  19960131 -0.030967  9.125000
1   10001  524    2281  19960229  0.009799  9.250000
2   10001  283    2309  19960331  0.032249  9.484375
3   10001  327    2309  19960430 -0.075440  8.812500
4   10001  103    2309  19960531 -0.025477  8.625000
5   10001  338    2321  19960630 -0.064290  8.000000
6   10001  207    2321  19960731  0.018937  8.187500
7   10001  507    2321  19960831  0.034068  8.500000
8   10001  205    2346  19960930  0.037365  8.750000
9   10001  203    2346  19961031 -0.032771  8.500000
清洗后的收益率数据维度: (2516385, 7)


,PERMNO,monthend,exret,mvel,VOL,price,mom1m
1,10001,1996-02-29,0.009799,2.109925e+04,524,9.250000,-0.030967
2,10001,1996-03-31,0.032249,2.189942e+04,283,9.484375,0.009799
3,10001,1996-04-30,-0.075440,2.034806e+04,327,8.812500,0.032249
4,10001,1996-05-31,-0.025477,1.991512e+04,103,8.625000,-0.075440
5,10001,1996-06-30,-0.064290,1.856800e+04,338,8.000000,-0.025477
...,...,...,...,...,...,...,...
2542365,93436,2023-08-31,-0.039462,8.191443e+08,25029170,258.079987,0.017122
2542366,93436,2023-09-30,-0.034756,7.954494e+08,24395440,250.220001,-0.039462
2542367,93436,2023-10-31,-0.202046,6.384545e+08,25905681,200.839996,-0.034756
2542368,93436,2023-11-30,0.190979,7.631954e+08,26395792,240.080002,-0.202046


In [3]:
# 1. 读取特征数据 (请确保文件名与你的本地文件一致)
osap_features = pd.read_csv('signed_predictors_dl_wide.csv')

# 为了代码鲁棒性，先统一把所有列名转成大写，避免大小写造成的匹配错误
osap_features.columns = [str(col).upper() for col in osap_features.columns]

# 2. 处理日期列 (将你的 YYYYMM 转为标准的月末 DATE)
if 'YYYYMM' in osap_features.columns:
    osap_features.rename(columns={'YYYYMM': 'DATE'}, inplace=True)

# 转换为 datetime 格式，并统一对齐到月末
osap_features['DATE'] = pd.to_datetime(osap_features['DATE'].astype(str), format='%Y%m') + pd.offsets.MonthEnd(0)

osap_features.head(10)

,PERMNO,DATE,AM,AOP,ABNORMALACCRUALS,ACCRUALS,ACCRUALSBM,ACTIVISM1,ACTIVISM2,ADEXP,...,RETCONGLOMERATE,ROAQ,SFE,SINALGO,SKEW1,STD_TURN,TANG,ZEROTRADE12M,ZEROTRADE1M,ZEROTRADE6M
0,10000,1986-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10000,1986-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.785175e-08,NaN
2,10000,1986-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.023392e-07,NaN
3,10000,1986-04-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.467463e-08,NaN
4,10000,1986-05-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.649551e-08,NaN
5,10000,1986-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.360224e-08,NaN
6,10000,1986-07-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000000e+00,2.065574
7,10000,1986-08-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.909091e+00,4.032001
8,10000,1986-09-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.591646e-08,3.968504
9,10000,1986-10-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.225312e-08,3.937500


In [4]:
# ==========================================
# 3. 动态提取特征列，并剔除重复计算的特征
# ==========================================
id_cols = ['PERMNO', 'DATE']

# 提取除主键外的所有列作为初始特征池 (也就是那 200 多个特征)
feature_cols = [col for col in osap_features.columns if col not in id_cols]

# 定义我们要剔除的“黑名单”（涵盖市值、短期反转、价格的常见拼写）
# 因为这些我们已经在 Stock_return 表里自己算好了
blacklist = ['MVEL1', 'MVE', 'SIZE', 'MARKETEQUITY', 'MOM1M', 'STREVM', 'PRC', 'PRICE']

# 找出确实存在于特征池中，且需要被剔除的列
cols_to_drop = [col for col in feature_cols if col in blacklist]

# 正式剔除它们
if cols_to_drop:
    osap_features = osap_features.drop(columns=cols_to_drop)
    feature_cols = [col for col in feature_cols if col not in cols_to_drop]
    print(f"✅ 成功剔除与我们自己计算重复的核心特征: {cols_to_drop}")
else:
    print("✅ OSAP 数据集中未发现需要剔除的重复核心特征（OSAP 官方已提前过滤）。")

print(f"🔥 最终保留了 {len(feature_cols)} 个有效特征，开始进行缺失值填补 (这可能会花一两分钟)...")

feature_cols

✅ OSAP 数据集中未发现需要剔除的重复核心特征（OSAP 官方已提前过滤）。
🔥 最终保留了 209 个有效特征，开始进行缺失值填补 (这可能会花一两分钟)...


['AM',
 'AOP',
 'ABNORMALACCRUALS',
 'ACCRUALS',
 'ACCRUALSBM',
 'ACTIVISM1',
 'ACTIVISM2',
 'ADEXP',
 'AGEIPO',
 'ANALYSTREVISION',
 'ANALYSTVALUE',
 'ANNOUNCEMENTRETURN',
 'ASSETGROWTH',
 'BM',
 'BMDEC',
 'BPEBM',
 'BETA',
 'BETAFP',
 'BETALIQUIDITYPS',
 'BETATAILRISK',
 'BIDASKSPREAD',
 'BOOKLEVERAGE',
 'BRANDINVEST',
 'CBOPERPROF',
 'CF',
 'CPVOLSPREAD',
 'CASH',
 'CASHPROD',
 'CHASSETTURNOVER',
 'CHEQ',
 'CHFORECASTACCRUAL',
 'CHINV',
 'CHINVIA',
 'CHNANALYST',
 'CHNNCOA',
 'CHNWC',
 'CHTAX',
 'CHANGEINRECOMMENDATION',
 'CITATIONSRD',
 'COMPEQUISS',
 'COMPOSITEDEBTISSUANCE',
 'CONSRECOMM',
 'CONVDEBT',
 'COSKEWACX',
 'COSKEWNESS',
 'CREDRATDG',
 'CUSTOMERMOMENTUM',
 'DEBTISSUANCE',
 'DELBREADTH',
 'DELCOA',
 'DELCOL',
 'DELDRC',
 'DELEQU',
 'DELFINL',
 'DELLTI',
 'DELNETFIN',
 'DIVINIT',
 'DIVOMIT',
 'DIVSEASON',
 'DIVYIELDST',
 'DOLVOL',
 'DOWNRECOMM',
 'EBM',
 'EP',
 'EARNSUPBIG',
 'EARNINGSCONSISTENCY',
 'EARNINGSFORECASTDISPARITY',
 'EARNINGSSTREAK',
 'EARNINGSSURPRISE',
 'ENT

In [ ]:
#import pandas as pd
#import numpy as np
#import gc

print("🚀 启动内存极简模式：读取特征数据...")
osap_features = pd.read_csv('signed_predictors_dl_wide.csv')
osap_features.columns = [str(col).upper() for col in osap_features.columns]

# 处理日期对齐
if 'YYYYMM' in osap_features.columns:
    osap_features.rename(columns={'YYYYMM': 'DATE'}, inplace=True)
osap_features['DATE'] = pd.to_datetime(osap_features['DATE'].astype(str), format='%Y%m') + pd.offsets.MonthEnd(0)

# ==========================================
# 1. 特征筛选与去重
# ==========================================
id_cols = ['PERMNO', 'DATE']
feature_cols = [col for col in osap_features.columns if col not in id_cols]
blacklist = ['MVEL1', 'MVE', 'SIZE', 'MARKETEQUITY', 'MOM1M', 'STREVM', 'PRC', 'PRICE']

cols_to_drop = [col for col in feature_cols if col in blacklist]
if cols_to_drop:
    osap_features.drop(columns=cols_to_drop, inplace=True)
    feature_cols = [col for col in feature_cols if col not in cols_to_drop]

print(f"🔥 保留了 {len(feature_cols)} 个特征。")

# ==========================================
# 2. 核心优化一：逐列降维 (释放 50% 内存)
# ==========================================
print("📉 正在将 float64 瘦身为 float32，释放内存中...")
for col in feature_cols:
    # 逐列转换，防止生成巨大的中间矩阵
    osap_features[col] = osap_features[col].astype(np.float32)

gc.collect() # 强制回收内存碎片

# ==========================================
# 3. 核心优化二：按月切片处理 (彻底抛弃 groupby)
# ==========================================
unique_dates = osap_features['DATE'].unique()
total_months = len(unique_dates)

print(f"⏱️ 开始按月切片极速清洗 (共 {total_months} 个月)...")

for i, date in enumerate(unique_dates):
    # 每隔 120 个月(10年)打印一次进度，让你安心
    if i % 120 == 0:
        date_str = pd.to_datetime(date).strftime('%Y-%m-%d')
        print(f"   -> 进度: 正在处理 {date_str} (第 {i}/{total_months} 个月)...")
    # 找到当前月份的所有行索引 (约 6000 行)
    mask = osap_features['DATE'] == date
    
    # 提取当月的 6000行 x 200列 数据矩阵 (只占区区几 MB 内存)
    month_data = osap_features.loc[mask, feature_cols]
    
    # 【填补】求出当月所有特征的中位数，并填补 NaN
    medians = month_data.median()
    month_data = month_data.fillna(medians)
    month_data = month_data.fillna(0) # 极端情况防线
    
    # 【排名与缩放】对当月数据进行排名并缩放至 [-1, 1]
    ranked = month_data.rank(pct=True)
    month_scaled = (ranked * 2 - 1).astype(np.float32)
    
    # 【无缝贴回】把算好的当前月数据贴回到原表中
    osap_features.loc[mask, feature_cols] = month_scaled

gc.collect()
print(f"🎉 极速优化版清洗大功告成！最终数据维度：{osap_features.shape}")
display(osap_features.head())

🚀 启动内存极简模式：读取特征数据...
🔥 保留了 209 个特征。
📉 正在将 float64 瘦身为 float32，释放内存中...
⏱️ 开始按月切片极速清洗 (共 1212 个月)...
   -> 进度: 正在处理 1986-01-31 (第 0/1212 个月)...
   -> 进度: 正在处理 1996-01-31 (第 120/1212 个月)...
   -> 进度: 正在处理 2006-01-31 (第 240/1212 个月)...
   -> 进度: 正在处理 2016-01-31 (第 360/1212 个月)...
   -> 进度: 正在处理 1933-06-30 (第 480/1212 个月)...
   -> 进度: 正在处理 1943-06-30 (第 600/1212 个月)...
   -> 进度: 正在处理 1953-06-30 (第 720/1212 个月)...
   -> 进度: 正在处理 1963-06-30 (第 840/1212 个月)...
   -> 进度: 正在处理 1973-06-30 (第 960/1212 个月)...
   -> 进度: 正在处理 1983-06-30 (第 1080/1212 个月)...
   -> 进度: 正在处理 2025-12-31 (第 1200/1212 个月)...
🎉 极速优化版清洗大功告成！最终数据维度：(5416424, 211)


,PERMNO,DATE,AM,AOP,ABNORMALACCRUALS,ACCRUALS,ACCRUALSBM,ACTIVISM1,ACTIVISM2,ADEXP,...,RETCONGLOMERATE,ROAQ,SFE,SINALGO,SKEW1,STD_TURN,TANG,ZEROTRADE12M,ZEROTRADE1M,ZEROTRADE6M
0,10000,1986-01-31,0.000147,0.000147,0.000147,0.000147,0.028508,0.000147,0.000147,0.000147,...,-0.000294,0.000147,0.000147,-0.007054,0.000147,0.000147,0.000147,-0.004702,0.000147,-0.001323
1,10000,1986-02-28,0.000147,0.000147,0.000147,0.000147,0.026829,0.000147,0.000147,0.000147,...,0.000733,0.000147,0.000147,-0.007330,0.000147,0.000147,0.000147,0.004398,-0.245565,-0.002346
2,10000,1986-03-31,0.000146,0.000146,0.000146,0.000146,0.028451,0.000146,0.000146,0.000146,...,-0.000875,0.000146,0.000146,-0.007003,0.000146,0.000146,0.000146,0.002334,0.153779,-0.003648
3,10000,1986-04-30,0.000146,0.000146,0.000146,0.000146,0.026925,0.000146,0.000146,0.000146,...,0.000291,0.000146,0.000146,-0.006840,0.000146,0.000146,0.000146,-0.003638,0.148887,-0.000437
4,10000,1986-05-31,0.000144,0.000144,0.000144,0.000144,0.026380,0.000144,0.000144,0.000144,...,-0.000288,0.000144,0.000144,-0.006775,0.000144,0.000144,0.000144,0.002306,0.170823,-0.007208


In [8]:
#将清洗好的特征数据保存到本地
osap_features.to_parquet('osap_features_cleaned.parquet', index=False)

In [10]:
osap_features = pd.read_parquet('osap_features_cleaned.parquet')
osap_features.head(10)

,PERMNO,DATE,AM,AOP,ABNORMALACCRUALS,ACCRUALS,ACCRUALSBM,ACTIVISM1,ACTIVISM2,ADEXP,...,RETCONGLOMERATE,ROAQ,SFE,SINALGO,SKEW1,STD_TURN,TANG,ZEROTRADE12M,ZEROTRADE1M,ZEROTRADE6M
0,10000,1986-01-31,0.000147,0.000147,0.000147,0.000147,0.028508,0.000147,0.000147,0.000147,...,-0.000294,0.000147,0.000147,-0.007054,0.000147,0.000147,0.000147,-0.004702,0.000147,-0.001323
1,10000,1986-02-28,0.000147,0.000147,0.000147,0.000147,0.026829,0.000147,0.000147,0.000147,...,0.000733,0.000147,0.000147,-0.007330,0.000147,0.000147,0.000147,0.004398,-0.245565,-0.002346
2,10000,1986-03-31,0.000146,0.000146,0.000146,0.000146,0.028451,0.000146,0.000146,0.000146,...,-0.000875,0.000146,0.000146,-0.007003,0.000146,0.000146,0.000146,0.002334,0.153779,-0.003648
3,10000,1986-04-30,0.000146,0.000146,0.000146,0.000146,0.026925,0.000146,0.000146,0.000146,...,0.000291,0.000146,0.000146,-0.006840,0.000146,0.000146,0.000146,-0.003638,0.148887,-0.000437
4,10000,1986-05-31,0.000144,0.000144,0.000144,0.000144,0.026380,0.000144,0.000144,0.000144,...,-0.000288,0.000144,0.000144,-0.006775,0.000144,0.000144,0.000144,0.002306,0.170823,-0.007208
5,10000,1986-06-30,0.000143,0.000143,0.000143,0.000143,0.028453,0.000143,0.000143,0.000143,...,-0.001716,0.000143,0.000143,-0.006720,0.000143,0.000143,0.000143,0.010009,-0.082928,0.013154
6,10000,1986-07-31,0.000144,0.000144,0.000144,0.000144,0.027035,0.000144,0.000144,0.000144,...,-0.001726,0.000144,0.000144,-0.006903,0.000144,0.000144,0.000144,0.000288,0.438309,0.248490
7,10000,1986-08-31,0.000143,0.000143,0.000143,0.000143,0.026802,0.000143,0.000143,0.000143,...,-0.001147,0.000143,0.000143,-0.006880,0.000143,0.000143,0.000143,-0.004586,0.402609,0.313602
8,10000,1986-09-30,0.000142,0.000142,0.000142,0.000142,0.026765,0.000142,0.000142,0.000142,...,0.000142,0.000142,0.000142,-0.006834,0.000142,0.000142,0.000142,-0.003844,-0.696185,0.301680
9,10000,1986-10-31,0.000141,0.000141,0.000141,0.000141,0.026086,0.000141,0.000141,0.000141,...,0.001692,0.000141,0.000141,-0.006768,0.000141,0.000141,0.000141,-0.000705,-0.732092,0.272561


In [11]:
#合并清洗后的storck_return和特征数据集
print("🚀 启动 209 全量特征的终极合并流水线：动态异步对齐...")

# ==========================================
# 1. 加载数据
# ==========================================
print("正在加载数据集...")
# 读取你保存的 209 个全量特征
osap_features = pd.read_parquet('osap_features_cleaned.parquet')

# 读取收益率数据并计算我们自己补齐的 2 个特征
returns_df = pd.read_csv('stock_return.csv')
returns_df['monthend'] = pd.to_datetime(returns_df['monthend'].astype(str)) + pd.offsets.MonthEnd(0)

returns_df['prc'] = returns_df['prc'].abs()
returns_df['mvel1'] = returns_df['prc'] * returns_df['SHROUT']
returns_df = returns_df.sort_values(['PERMNO', 'monthend'])
returns_df['mom1m'] = returns_df.groupby('PERMNO')['exret'].shift(1)

# 对这 2 个特征进行当月截面缩放
crsp_feats = ['mvel1', 'mom1m']
returns_df[crsp_feats] = returns_df.groupby('monthend')[crsp_feats].transform(lambda x: x.rank(pct=True) * 2 - 1).astype(np.float32)


# ==========================================
# 2. 动态特征频率分类 (基于关键字规则)
# ==========================================
all_features = [col for col in osap_features.columns if col not in ['PERMNO', 'DATE']]

# 交易/价格/动量类关键字 (月度)
monthly_keys = ['MOM', 'VOL', 'RET', 'BETA', 'TURN', 'LIQ', 'SPREAD', 'PRICE', 'TRADE', 'SKEW', 'DELAY']
# 财报盈利类关键字 (季度)
quarterly_keys = ['ROA', 'ROE', 'EARN', 'SURP', 'EPS', 'PROF']

monthly_feats = []
quarterly_feats = []
annual_feats = []

# 自动扫描 209 个特征并分类
for f in all_features:
    if any(k in f for k in monthly_keys):
        monthly_feats.append(f)
    elif any(k in f for k in quarterly_keys):
        quarterly_feats.append(f)
    else:
        annual_feats.append(f) # 保守策略：不确定的全部作为年度处理，防止前瞻偏差

print(f"✅ 特征自动分类完毕: 月度 {len(monthly_feats)} 个, 季度 {len(quarterly_feats)} 个, 年度 {len(annual_feats)} 个.")


# ==========================================
# 3. 异步对齐合并 (Asynchronous Merging)
# ==========================================
# 构建目标变量的主干面板 (包含 t+1 的 exret)
final_panel = returns_df[['PERMNO', 'monthend', 'exret', 'mvel1', 'mom1m']].copy()
# CRSP特征是月度，滞后 1 个月
final_panel['merge_date'] = final_panel['monthend'] + pd.DateOffset(months=1) + pd.offsets.MonthEnd(0)

# ----- A. 合并月度特征 (Lag = 1) -----
print("合并月度特征 (Lag 1)...")
df_m = osap_features[['PERMNO', 'DATE'] + monthly_feats].copy()
df_m['DATE'] = df_m['DATE'] + pd.DateOffset(months=1) + pd.offsets.MonthEnd(0)
final_panel = pd.merge(final_panel, df_m, left_on=['PERMNO', 'merge_date'], right_on=['PERMNO', 'DATE'], how='left')
final_panel.drop(columns=['DATE'], inplace=True)
del df_m; gc.collect()

# ----- B. 合并季度特征 (Lag = 4) -----
print("合并季度特征 (Lag 4)...")
df_q = osap_features[['PERMNO', 'DATE'] + quarterly_feats].copy()
df_q['DATE'] = df_q['DATE'] + pd.DateOffset(months=4) + pd.offsets.MonthEnd(0)
final_panel = pd.merge(final_panel, df_q, left_on=['PERMNO', 'merge_date'], right_on=['PERMNO', 'DATE'], how='left')
final_panel.drop(columns=['DATE'], inplace=True)
del df_q; gc.collect()

# ----- C. 合并年度特征 (Lag = 6) -----
print("合并年度特征 (Lag 6)...")
df_a = osap_features[['PERMNO', 'DATE'] + annual_feats].copy()
df_a['DATE'] = df_a['DATE'] + pd.DateOffset(months=6) + pd.offsets.MonthEnd(0)
final_panel = pd.merge(final_panel, df_a, left_on=['PERMNO', 'merge_date'], right_on=['PERMNO', 'DATE'], how='left')
final_panel.drop(columns=['DATE', 'merge_date'], inplace=True)
del df_a; gc.collect()


# ==========================================
# 4. 特征延续：向前填充 (Forward Fill)
# ==========================================
print("正在执行特征有效期填充 (季度延续 2 个月，年度延续 11 个月)...")
final_panel = final_panel.sort_values(['PERMNO', 'monthend'])

# 季度特征：一份季报可以用 3 个月（合并当月 + 往后延 2 个月）
final_panel[quarterly_feats] = final_panel.groupby('PERMNO')[quarterly_feats].ffill(limit=2)

# 年度特征：一份年报可以用 12 个月（合并当月 + 往后延 11 个月）
final_panel[annual_feats] = final_panel.groupby('PERMNO')[annual_feats].ffill(limit=11)

# 清除目标变量缺失或月度核心特征缺失的行
final_panel = final_panel.dropna(subset=['exret'] + monthly_feats) 
# 由于某些极早年份财报缺失，ffill后可能仍有NaN，统一用0（中位数）兜底保证数据完美
final_panel = final_panel.fillna(0)


# ==========================================
# 5. 保存终极训练集
# ==========================================
final_panel.to_parquet('GKX_209Features_Final_Panel.parquet', index=False)
gc.collect()

print(f"🎉 209 全量特征异步合并大功告成！拥有 {final_panel.shape[1]-3} 个预测因子的超级面板已就绪。最终维度: {final_panel.shape}")
display(final_panel.head())

🚀 启动 209 全量特征的终极合并流水线：动态异步对齐...
正在加载数据集...
✅ 特征自动分类完毕: 月度 64 个, 季度 14 个, 年度 131 个.
合并月度特征 (Lag 1)...
合并季度特征 (Lag 4)...
合并年度特征 (Lag 6)...
正在执行特征有效期填充 (季度延续 2 个月，年度延续 11 个月)...
🎉 209 全量特征异步合并大功告成！拥有 211 个预测因子的超级面板已就绪。最终维度: (2542370, 214)


,PERMNO,monthend,exret,mvel1,mom1m,ANNOUNCEMENTRETURN,BETA,BETAFP,BETALIQUIDITYPS,BETATAILRISK,...,CFP,DNOA,FGR5YRLAG,GRCAPX,GRCAPX3Y,HIRE,REALESTATE,SFE,SINALGO,TANG
0,10001,1996-01-31,-0.030967,-0.608577,0.000000,-0.583582,-0.961602,-0.864945,0.657288,-0.860973,...,0.776815,0.334321,0.980166,-0.714807,-0.722786,-0.677647,0.000114,0.000228,-0.009347,0.000114
1,10001,1996-02-29,0.009799,-0.617309,-0.382118,-0.412538,-0.962297,-0.887111,0.647085,-0.860368,...,0.768873,0.340674,0.980247,-0.715972,-0.725281,-0.677830,0.000114,0.000227,-0.009309,0.000114
2,10001,1996-03-31,0.032249,-0.613162,0.055840,-0.436620,-0.960563,-0.844637,0.626219,-0.861538,...,0.764746,0.345438,0.980377,-0.720311,-0.727078,-0.679035,0.000113,0.000226,-0.009361,0.000113
3,10001,1996-04-30,-0.075440,-0.647527,0.294048,-0.470936,-0.966262,-0.858601,0.584829,-0.858386,...,0.744111,0.355141,0.980574,-0.720889,-0.725131,-0.679134,0.000112,0.000223,-0.009378,0.000112
4,10001,1996-05-31,-0.025477,-0.681069,-0.747561,-0.315862,-0.969231,-0.842546,0.575809,-0.852308,...,0.799734,-0.439521,0.980727,-0.696500,-0.842047,-0.459681,0.000111,0.000222,-0.009415,0.000111


In [12]:
final_panel = pd.read_parquet('GKX_209Features_Final_Panel.parquet')
final_panel

,PERMNO,monthend,exret,mvel1,mom1m,ANNOUNCEMENTRETURN,BETA,BETAFP,BETALIQUIDITYPS,BETATAILRISK,...,CFP,DNOA,FGR5YRLAG,GRCAPX,GRCAPX3Y,HIRE,REALESTATE,SFE,SINALGO,TANG
0,10001,1996-01-31,-0.030967,-0.608577,0.000000,-0.583582,-0.961602,-0.864945,0.657288,-0.860973,...,0.776815,0.334321,0.980166,-0.714807,-0.722786,-0.677647,0.000114,0.000228,-0.009347,0.000114
1,10001,1996-02-29,0.009799,-0.617309,-0.382118,-0.412538,-0.962297,-0.887111,0.647085,-0.860368,...,0.768873,0.340674,0.980247,-0.715972,-0.725281,-0.677830,0.000114,0.000227,-0.009309,0.000114
2,10001,1996-03-31,0.032249,-0.613162,0.055840,-0.436620,-0.960563,-0.844637,0.626219,-0.861538,...,0.764746,0.345438,0.980377,-0.720311,-0.727078,-0.679035,0.000113,0.000226,-0.009361,0.000113
3,10001,1996-04-30,-0.075440,-0.647527,0.294048,-0.470936,-0.966262,-0.858601,0.584829,-0.858386,...,0.744111,0.355141,0.980574,-0.720889,-0.725131,-0.679134,0.000112,0.000223,-0.009378,0.000112
4,10001,1996-05-31,-0.025477,-0.681069,-0.747561,-0.315862,-0.969231,-0.842546,0.575809,-0.852308,...,0.799734,-0.439521,0.980727,-0.696500,-0.842047,-0.459681,0.000111,0.000222,-0.009415,0.000111
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2542365,93436,2023-08-31,-0.039462,0.999142,-0.113555,-0.766625,0.771789,0.869475,0.608426,-0.791408,...,-0.615470,-0.720818,-0.993957,-0.930507,-0.937758,-0.877530,0.000101,0.000101,-0.004129,-0.788498
2542366,93436,2023-09-30,-0.034756,0.999144,-0.064985,-0.770848,0.772284,0.880398,0.732075,-0.802646,...,-0.605938,-0.719451,-0.993941,-0.930115,-0.937184,-0.877197,0.000101,0.000101,-0.004242,-0.787518
2542367,93436,2023-10-31,-0.202046,0.998500,0.221235,-0.868936,0.823685,0.883612,-0.401345,-0.787199,...,-0.617807,-0.720170,-0.993943,-0.930143,-0.937008,-0.877246,0.000101,0.000101,-0.004139,-0.787603
2542368,93436,2023-11-30,0.190979,0.998722,-0.782632,-0.852567,0.829792,0.886324,-0.319980,0.753533,...,-0.577922,-0.755479,-0.993506,-0.823661,-0.881494,-0.866883,0.000101,0.000101,-0.004363,-0.784700


In [ ]:
osap_sample = pd.read_csv('signed_predictors_dl_wide.csv', nrows=5)

# 获取所有列名并转为列表
osap_columns = osap_sample.columns.tolist()

print(f"OSAP 数据集总共包含 {len(osap_columns)} 列！")
print("以下是完整的表头列表：")
# 每 10 个换一行打印，方便查看
for i in range(0, len(osap_columns), 10):
    print(osap_columns[i:i+10])